In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

In [2]:
HEADERS = {"User-Agent": "Mozilla/5.0 Chrome/124.0"}

In [3]:
r = requests.get("https://www.parcs-france.com/carte/", headers=HEADERS)
soup = BeautifulSoup(r.text, "html.parser")

tbody = soup.find("tbody", {"class": "row-striping row-hover"})
rows = tbody.find_all("tr")

data = []

for row in rows:
    name_tag = row.find("a")  # peut être None

    name = name_tag.get_text(strip=True) if name_tag else row.get_text(strip=True)
    href = name_tag.get("href") if name_tag else None

    data.append({
        "name": name,
        "href": href
    })

df = pd.DataFrame(data)

In [12]:
def extract_info(url):

    if pd.isna(url):
        return {
            "adresse": None,
            "latitude": None,
            "longitude": None
        }

    r = requests.get(url, headers=HEADERS, timeout=15)
    soup = BeautifulSoup(r.text, "html.parser")

    adresse = None
    lat, lon = None, None

    for strong in soup.find_all("strong"):
        if "Adresse" in strong.get_text():
            next_node = strong.next_sibling
            if next_node:
                adresse = str(next_node).strip().lstrip(": ").strip()
            break

    gps_match = re.search(
        r"latitude\s*([-\d.]+)\s*\|\s*longitude\s*([-\d.]+)",
        soup.get_text(),
        re.IGNORECASE
    )


    if gps_match:
        lat = float(gps_match.group(1))
        lon = float(gps_match.group(2))

    return {"adresse": adresse, "latitude": lat, "longitude": lon}

In [13]:
df_info = df["href"].apply(extract_info).apply(pd.Series)

In [14]:
parc_attraction = pd.concat([df, df_info], axis=1)

In [15]:
parc_attraction.to_csv("parc attractions.csv")